# Teste do pipeline Datajud — ano 2009

Notebook isolado para validar a conexão com a API, a transformação dos registros e a gravação em Parquet.

O teste coleta o ano 2009 inteiro e grava em caminhos separados. A coleta é retomável: após cada página, o checkpoint registra o cursor da API, a quantidade de processos e o tamanho confirmado do arquivo bruto.

- bruto e checkpoint: `data/raw/datajud_teste/2009/`;
- Parquet: `data/processed/datajud/teste/tabela_principal_helios_2009_teste.parquet`.

O arquivo oficial de 2025–2026 e o notebook principal não são alterados.

In [1]:
from pathlib import Path
import csv
import json
import os
import re
import time
import requests

def localizar_projeto():
    configurado = os.getenv('HELIO_PROJECT_ROOT')
    atual = Path(configurado).expanduser().resolve() if configurado else Path.cwd().resolve()
    for raiz in (atual, *atual.parents):
        if (raiz / 'manage.py').is_file():
            return raiz
    raise RuntimeError('Execute o notebook dentro do projeto Hélio.')

def carregar_env(caminho):
    if not caminho.is_file():
        return
    for linha in caminho.read_text(encoding='utf-8').splitlines():
        linha = linha.strip()
        if not linha or linha.startswith('#') or '=' not in linha:
            continue
        chave, valor = linha.split('=', 1)
        chave = chave.strip().lstrip(chr(65279))
        valor = valor.strip().strip(chr(34)).strip(chr(39))
        if not os.environ.get(chave):
            os.environ[chave] = valor

PROJECT_ROOT = localizar_projeto()
carregar_env(PROJECT_ROOT / '.env')
API_KEY = os.getenv('DATAJUD_API_KEY', '').strip()
if not API_KEY:
    raise RuntimeError('DATAJUD_API_KEY não encontrada no .env.')

ANO = 2009
LIMITE_POR_ANO = None  # coleta o ano 2009 inteiro e retoma pelo checkpoint
URL_DATAJUD = 'https://api-publica.datajud.cnj.jus.br/api_publica_tjba/_search'
LOTE_INICIAL = 5_000
LOTE_MINIMO = 2_500
LOTE_MAXIMO = 10_000
LOTE_PASSO_SUCESSO = 500
LOTE_PASSO_FALHA = 500
LOTE_PASSO_CONGESTIONAMENTO = 1_000
SUCESSOS_PARA_AUMENTAR = 3
RETRY_INICIAL = 12
RETRY_MINIMO = 9
RETRY_MAXIMO = 15
RETRY_PASSO = 3
TIMEOUT_CONEXAO = 30
TIMEOUT_LEITURA = 300
STATUS_CONGESTIONAMENTO = {429, 502, 503, 504}
CHECKPOINT_VERSION = 7
LOTE = LOTE_INICIAL
ESPERA_RETRY_ATUAL = RETRY_INICIAL
SUCESSOS_CONSECUTIVOS = 0
PASTA_RAW = PROJECT_ROOT / 'data' / 'raw' / 'datajud_teste' / str(ANO)
ARQUIVO_PARQUET = PROJECT_ROOT / 'data' / 'processed' / 'datajud' / 'teste' / f'tabela_principal_helios_{ANO}_teste.parquet'
PASTA_RAW.mkdir(parents=True, exist_ok=True)
ARQUIVO_PARQUET.parent.mkdir(parents=True, exist_ok=True)

def localizar_dicionario_municipios():
    configurado = os.getenv('MUNICIPIOS_BAHIA_CSV')
    candidatos = [
        Path(configurado).expanduser() if configurado else None,
        PROJECT_ROOT / 'data' / 'raw' / 'ibge' / 'municipios_bahia.csv',
        Path.home() / 'Downloads' / 'municipios_bahia.csv',
    ]
    for candidato in candidatos:
        if candidato and candidato.is_file():
            return candidato.resolve()
    raise FileNotFoundError('Dicionário de municípios não encontrado. Defina MUNICIPIOS_BAHIA_CSV.')

MUNICIPIOS_CSV = localizar_dicionario_municipios()
H_MAPA_MUNICIPIOS = {}
with MUNICIPIOS_CSV.open('r', encoding='utf-8-sig', newline='') as arquivo:
    leitor = csv.DictReader(arquivo)
    campos = {campo.strip().upper(): campo for campo in leitor.fieldnames or []}
    coluna_codigo = campos.get('CÓDIGO IBGE') or campos.get('CODIGO_IBGE')
    coluna_nome = campos.get('MUNICÍPIO') or campos.get('MUNICIPIO')
    if not coluna_codigo or not coluna_nome:
        raise ValueError(f'Colunas incompatíveis no dicionário: {leitor.fieldnames}')
    for registro in leitor:
        codigo = str(registro.get(coluna_codigo) or '').strip()
        nome = str(registro.get(coluna_nome) or '').strip().upper()
        if re.fullmatch(r'\d{7}', codigo) and nome:
            H_MAPA_MUNICIPIOS[codigo] = nome

session = requests.Session()
session.headers.update({
    'Authorization': f'ApiKey {API_KEY}',
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'Accept-Encoding': 'gzip',
})

print(f'Projeto: {PROJECT_ROOT}')
print(f'Ano: {ANO}')
print(f'Limite de teste: {LIMITE_POR_ANO}')
print(f'Parquet de teste: {ARQUIVO_PARQUET}')
print(f'Dicionário de municípios: {MUNICIPIOS_CSV} ({len(H_MAPA_MUNICIPIOS):,} entradas)')

Projeto: C:\Users\drnsarmento\Helio
Ano: 2000
Limite de teste: 1000
Parquet de teste: C:\Users\drnsarmento\Helio\data\processed\datajud\teste\tabela_principal_helios_2000_teste.parquet


In [2]:
SOURCE_FIELDS = [
    'grau', 'numeroProcesso', 'orgaoJulgador.codigo', 'orgaoJulgador.nome',
    'orgaoJulgador.codigoMunicipioIBGE', 'classe.codigo', 'classe.nome',
    'assuntos.codigo', 'assuntos.nome',
]
LIMITES = {'grau': 2, 'processo': 25, 'id_municipio': 7, 'municipio': 27, 'nome_classe': 310, 'codigo_orgao': 5, 'nome_orgao': 155}

def codigo_numerico(valor):
    texto = str(valor).strip()
    return int(texto) if texto.isdigit() else texto

def json_lista(valores):
    return json.dumps(valores, ensure_ascii=False)

def formatar_processo(valor):
    bruto = str(valor or '').strip()
    if re.fullmatch(r'\d{7}-\d{2}\.\d{4}\.\d\.\d{2}\.\d{4}', bruto):
        return bruto
    if re.fullmatch(r'\d{1,20}', bruto):
        numero = bruto.zfill(20)
        return f'{numero[:7]}-{numero[7:9]}.{numero[9:13]}.{numero[13]}.{numero[14:16]}.{numero[16:20]}'
    if re.fullmatch(r'sigiloso\(.*\)', bruto, flags=re.IGNORECASE):
        return bruto
    raise ValueError(f'Número de processo inválido: {bruto!r}')

def transformar(hit):
    source = hit.get('_source') or {}
    orgao = source.get('orgaoJulgador') or {}
    classe = source.get('classe') or {}
    assuntos = source.get('assuntos') or []
    if isinstance(assuntos, dict):
        assuntos = [assuntos]
    assuntos = [item for item in assuntos if isinstance(item, dict) and item.get('codigo') is not None]
    codigo_municipio = str(orgao.get('codigoMunicipioIBGE') or '').strip()
    codigos_classe = [codigo_numerico(classe['codigo'])] if classe.get('codigo') is not None else []
    nomes_classe = [str(classe.get('nome') or '')] if codigos_classe else []
    codigos_assunto = [codigo_numerico(item['codigo']) for item in assuntos]
    nomes_assunto = [str(item.get('nome') or '') for item in assuntos]
    registro = {
        'ano': str(ANO), 'grau': str(source.get('grau') or ''),
        'processo': formatar_processo(source.get('numeroProcesso')),
        'id_municipio': codigo_municipio,
        'municipio': H_MAPA_MUNICIPIOS.get(codigo_municipio, ''),
        'codigo_classe': json_lista(codigos_classe),
        'nome_classe': json_lista(nomes_classe),
        'codigo_assunto': json_lista(codigos_assunto),
        'nome_assunto': json_lista(nomes_assunto),
        'codigo_orgao': str(orgao.get('codigo') or ''),
        'nome_orgao': str(orgao.get('nome') or ''),
    }
    for campo, limite in LIMITES.items():
        if len(registro[campo]) > limite:
            raise ValueError(f'{campo} excede o limite do modelo: {len(registro[campo])} > {limite}')
    return registro

def post_retry(payload, contexto):
    global LOTE, ESPERA_RETRY_ATUAL, SUCESSOS_CONSECUTIVOS
    tentativa = 0
    while True:
        tentativa += 1
        lote_usado = LOTE
        requisicao = dict(payload)
        requisicao['size'] = lote_usado
        params = {
            'allow_partial_search_results': 'false',
            'filter_path': 'timed_out,_shards.failed,hits.hits._source,hits.hits.sort',
        }
        erro_curto = None
        congestionamento = False
        try:
            inicio = time.perf_counter()
            resposta = session.post(URL_DATAJUD, json=requisicao, params=params, timeout=(TIMEOUT_CONEXAO, TIMEOUT_LEITURA))
            duracao = time.perf_counter() - inicio
            if resposta.status_code != 200:
                erro_curto = f'HTTP {resposta.status_code}'
                congestionamento = resposta.status_code in STATUS_CONGESTIONAMENTO
            else:
                dados = resposta.json()
                if not isinstance(dados, dict):
                    erro_curto = 'JSON inválido'
                elif dados.get('timed_out'):
                    erro_curto = 'timeout da API'
                    congestionamento = True
                else:
                    shards = dados.get('_shards') or {}
                    falhos = int(shards.get('failed', 0) or 0) if isinstance(shards, dict) else 1
                    hits_obj = dados.get('hits')
                    hits = hits_obj.get('hits') if isinstance(hits_obj, dict) else None
                    if falhos == 0 and not isinstance(hits, list):
                        verificacao = session.post(URL_DATAJUD, json=requisicao, params={'allow_partial_search_results': 'false'}, timeout=(TIMEOUT_CONEXAO, TIMEOUT_LEITURA))
                        if verificacao.status_code == 200:
                            integral = verificacao.json()
                            if integral.get('timed_out'):
                                erro_curto = 'timeout da API'
                                congestionamento = True
                                falhos = 1
                            else:
                                integral_shards = integral.get('_shards') or {}
                                falhos = int(integral_shards.get('failed', 0) or 0)
                                dados = integral
                                hits_obj = integral.get('hits')
                                hits = hits_obj.get('hits') if isinstance(hits_obj, dict) else None
                        else:
                            erro_curto = f'HTTP {verificacao.status_code}'
                            congestionamento = verificacao.status_code in STATUS_CONGESTIONAMENTO
                    if falhos:
                        erro_curto = f'{falhos} shards com falha'
                        congestionamento = True
                    elif not isinstance(hits, list):
                        erro_curto = 'resposta sem hits.hits'
                    elif hits and (not isinstance(hits[-1], dict) or not hits[-1].get('sort')):
                        erro_curto = 'página sem sort'
                    else:
                        ESPERA_RETRY_ATUAL = max(RETRY_MINIMO, ESPERA_RETRY_ATUAL - RETRY_PASSO)
                        SUCESSOS_CONSECUTIVOS += 1
                        if SUCESSOS_CONSECUTIVOS >= SUCESSOS_PARA_AUMENTAR:
                            LOTE = min(LOTE_MAXIMO, LOTE + LOTE_PASSO_SUCESSO)
                            SUCESSOS_CONSECUTIVOS = 0
                        return dados, duracao, lote_usado
        except requests.exceptions.Timeout:
            erro_curto = 'timeout de conexão/leitura'
            congestionamento = True
        except requests.exceptions.RequestException as erro:
            erro_curto = type(erro).__name__
        except (ValueError, TypeError, KeyError) as erro:
            erro_curto = type(erro).__name__
        SUCESSOS_CONSECUTIVOS = 0
        queda = LOTE_PASSO_CONGESTIONAMENTO if congestionamento else LOTE_PASSO_FALHA
        LOTE = max(LOTE_MINIMO, LOTE - queda)
        ESPERA_RETRY_ATUAL = min(RETRY_MAXIMO, ESPERA_RETRY_ATUAL + RETRY_PASSO)
        if tentativa <= 3 or tentativa % 10 == 0:
            print(f'[retry] {contexto} | {erro_curto} | tentativa {tentativa} | lote {LOTE:,} | espera {ESPERA_RETRY_ATUAL}s')
        time.sleep(ESPERA_RETRY_ATUAL)

def buscar_pagina(search_after=None):
    payload = {
        'track_total_hits': False, '_source': SOURCE_FIELDS,
        'query': {'range': {'dataAjuizamento': {'gte': f'{ANO}0101000000', 'lt': f'{ANO + 1}0101000000'}}},
        'sort': [{'id.keyword': {'order': 'asc'}}],
    }
    if search_after is not None:
        payload['search_after'] = search_after
    return post_retry(payload, str(ANO))


In [3]:
ARQUIVO_PARCIAL = PASTA_RAW / f'tjba_{ANO}_teste.jsonl.part'
ARQUIVO_FINAL = PASTA_RAW / f'tjba_{ANO}_teste.jsonl'
CHECKPOINT = PASTA_RAW / f'checkpoint_{ANO}_teste.json'

def salvar_checkpoint(estado):
    temporario = CHECKPOINT.with_suffix('.tmp')
    with temporario.open('w', encoding='utf-8') as arquivo:
        json.dump(estado, arquivo, ensure_ascii=False, indent=2)
        arquivo.flush()
        os.fsync(arquivo.fileno())
    os.replace(temporario, CHECKPOINT)

def coletar_teste():
    global LOTE, ESPERA_RETRY_ATUAL, SUCESSOS_CONSECUTIVOS
    if ARQUIVO_FINAL.is_file() and ARQUIVO_FINAL.stat().st_size > 0:
        print(f'Coleta já existente: {ARQUIVO_FINAL}')
        return ARQUIVO_FINAL
    if CHECKPOINT.is_file():
        estado = json.loads(CHECKPOINT.read_text(encoding='utf-8'))
    elif ARQUIVO_PARCIAL.is_file() and ARQUIVO_PARCIAL.stat().st_size > 0:
        raise RuntimeError('Existe arquivo parcial sem checkpoint.')
    else:
        estado = {'version': CHECKPOINT_VERSION, 'ano': ANO, 'search_after': None, 'processos': 0, 'paginas': 0, 'file_size': 0, 'concluido': False}
    if int(estado.get('ano', ANO)) != ANO or int(estado.get('version', CHECKPOINT_VERSION)) != CHECKPOINT_VERSION:
        raise RuntimeError(f'Checkpoint incompatível: esperado o ano {ANO}.')
    tamanho_checkpoint = int(estado.get('file_size', 0))
    tamanho_atual = ARQUIVO_PARCIAL.stat().st_size if ARQUIVO_PARCIAL.is_file() else 0
    if tamanho_atual < tamanho_checkpoint:
        raise RuntimeError('O arquivo parcial é menor que o checkpoint; não é seguro continuar.')
    if tamanho_atual > tamanho_checkpoint:
        with ARQUIVO_PARCIAL.open('r+b') as arquivo:
            arquivo.truncate(tamanho_checkpoint)
    if estado.get('concluido'):
        if not ARQUIVO_PARCIAL.is_file():
            raise RuntimeError('Checkpoint concluído sem o arquivo parcial correspondente.')
        ARQUIVO_PARCIAL.replace(ARQUIVO_FINAL)
        return ARQUIVO_FINAL
    total = int(estado.get('processos', 0))
    search_after = estado.get('search_after')
    pagina = int(estado.get('paginas', 0))
    if LIMITE_POR_ANO and total >= LIMITE_POR_ANO:
        return ARQUIVO_PARCIAL
    LOTE = LOTE_INICIAL
    ESPERA_RETRY_ATUAL = RETRY_INICIAL
    SUCESSOS_CONSECUTIVOS = 0
    print(f'Lote adaptativo: {LOTE:,} | mínimo {LOTE_MINIMO:,} | máximo {LOTE_MAXIMO:,}')
    concluido = False
    with ARQUIVO_PARCIAL.open('ab') as arquivo:
        while True:
            dados, duracao, lote_usado = buscar_pagina(search_after)
            hits = (dados.get('hits') or {}).get('hits')
            if not isinstance(hits, list):
                raise RuntimeError('Resposta sem hits.hits após retry.')
            if not hits:
                arquivo.flush()
                os.fsync(arquivo.fileno())
                estado.update({'version': CHECKPOINT_VERSION, 'ano': ANO, 'search_after': search_after, 'processos': total, 'paginas': pagina, 'file_size': arquivo.tell(), 'concluido': True})
                salvar_checkpoint(estado)
                concluido = True
                break
            novo_search_after = hits[-1].get('sort') if isinstance(hits[-1], dict) else None
            if not novo_search_after or novo_search_after == search_after:
                raise RuntimeError('search_after não avançou.')
            bloco = ('\n'.join(json.dumps(transformar(hit), ensure_ascii=False, separators=(',', ':')) for hit in hits) + '\n').encode('utf-8')
            arquivo.write(bloco)
            arquivo.flush()
            os.fsync(arquivo.fileno())
            pagina += 1
            total += len(hits)
            search_after = novo_search_after
            estado.update({'version': CHECKPOINT_VERSION, 'ano': ANO, 'search_after': search_after, 'processos': total, 'paginas': pagina, 'file_size': arquivo.tell(), 'concluido': False})
            salvar_checkpoint(estado)
            print(f'{ANO} | página {pagina:,} | recebidos {len(hits):,}/{lote_usado:,} | {duracao:.2f}s | total {total:,} | próximo lote {LOTE:,}')
            if LIMITE_POR_ANO and total >= LIMITE_POR_ANO:
                print('Limite de teste atingido; o arquivo parcial foi preservado.')
                return ARQUIVO_PARCIAL
    if concluido:
        ARQUIVO_PARCIAL.replace(ARQUIVO_FINAL)
        print(f'Ano {ANO} concluído: {total:,} registros')
        return ARQUIVO_FINAL

coletar_teste()

WindowsPath('C:/Users/drnsarmento/Helio/data/raw/datajud_teste/2000/tjba_2000_teste.jsonl.part')

In [4]:
import pyarrow as pa
import pyarrow.parquet as pq

COLUNAS = ['ano', 'grau', 'processo', 'id_municipio', 'municipio', 'codigo_classe', 'nome_classe', 'codigo_assunto', 'nome_assunto', 'codigo_orgao', 'nome_orgao']
SCHEMA = pa.schema([pa.field(campo, pa.string()) for campo in COLUNAS])

def normalizar_lista(valor, numerico=False):
    if valor is None:
        valores = []
    elif isinstance(valor, list):
        valores = valor
    elif isinstance(valor, str):
        texto = valor.strip()
        if texto.startswith('[') and texto.endswith(']'):
            try:
                valores = json.loads(texto)
            except json.JSONDecodeError:
                valores = [texto]
            if not isinstance(valores, list):
                valores = [valores]
        else:
            valores = [parte.strip() for parte in texto.split('|')]
    else:
        valores = [valor]
    if numerico:
        return [codigo_numerico(item) for item in valores if str(item).strip()]
    return [str(item).strip() for item in valores if str(item).strip()]

def normalizar_registro(registro):
    id_municipio = str(registro.get('id_municipio') or '').strip()
    municipio = str(registro.get('municipio') or '').strip().upper() or H_MAPA_MUNICIPIOS.get(id_municipio, '')
    normalizado = {
        'ano': str(registro.get('ano') or ANO),
        'grau': str(registro.get('grau') or ''),
        'processo': formatar_processo(registro.get('processo')),
        'id_municipio': id_municipio,
        'municipio': municipio,
        'codigo_classe': json_lista(normalizar_lista(registro.get('codigo_classe'), numerico=True)),
        'nome_classe': json_lista(normalizar_lista(registro.get('nome_classe'))),
        'codigo_assunto': json_lista(normalizar_lista(registro.get('codigo_assunto'), numerico=True)),
        'nome_assunto': json_lista(normalizar_lista(registro.get('nome_assunto'))),
        'codigo_orgao': str(registro.get('codigo_orgao') or ''),
        'nome_orgao': str(registro.get('nome_orgao') or ''),
    }
    return {campo: str(normalizado.get(campo) or '') for campo in COLUNAS}

def consolidar_teste():
    if ARQUIVO_FINAL.is_file():
        caminho = ARQUIVO_FINAL
        limite = caminho.stat().st_size
    else:
        caminho = ARQUIVO_PARCIAL
        limite = json.loads(CHECKPOINT.read_text(encoding='utf-8'))['file_size']
    temporario = Path(str(ARQUIVO_PARQUET) + '.tmp')
    lote = []
    total = 0
    with pq.ParquetWriter(temporario, SCHEMA, compression='snappy') as escritor:
        with caminho.open('rb') as arquivo:
            while arquivo.tell() < limite:
                linha = arquivo.readline(limite - arquivo.tell())
                if not linha:
                    break
                lote.append(normalizar_registro(json.loads(linha.decode('utf-8'))))
                total += 1
                if len(lote) >= 1_000:
                    escritor.write_table(pa.Table.from_pylist(lote, schema=SCHEMA))
                    lote.clear()
        if lote:
            escritor.write_table(pa.Table.from_pylist(lote, schema=SCHEMA))
    validacao = pq.ParquetFile(temporario)
    linhas_validadas = validacao.metadata.num_rows
    validacao.close()
    if linhas_validadas != total:
        temporario.unlink(missing_ok=True)
        raise RuntimeError('Quantidade de linhas inválida no Parquet de teste.')
    temporario.replace(ARQUIVO_PARQUET)
    print(f'Parquet de teste salvo: {ARQUIVO_PARQUET}')
    print(f'Registros: {total:,}')
    return ARQUIVO_PARQUET

consolidar_teste()

Parquet de teste salvo: C:\Users\drnsarmento\Helio\data\processed\datajud\teste\tabela_principal_helios_2000_teste.parquet
Registros: 1,000


WindowsPath('C:/Users/drnsarmento/Helio/data/processed/datajud/teste/tabela_principal_helios_2000_teste.parquet')

In [5]:
parquet = pq.ParquetFile(ARQUIVO_PARQUET)
print(f'Arquivo: {ARQUIVO_PARQUET}')
print(f'Linhas: {parquet.metadata.num_rows:,}')
print('Colunas:', parquet.schema_arrow.names)
amostra = parquet.read_row_group(0).slice(0, 5).to_pylist()
parquet.close()
print('Amostra:', amostra)
print('Teste concluído. Para validar o importador sem gravar no banco:')
print(f'python manage.py importar_datajud --arquivo {ARQUIVO_PARQUET} --dry-run')

Arquivo: C:\Users\drnsarmento\Helio\data\processed\datajud\teste\tabela_principal_helios_2000_teste.parquet
Linhas: 1,000
Colunas: ['ano', 'grau', 'processo', 'id_municipio', 'municipio', 'codigo_classe', 'nome_classe', 'codigo_assunto', 'nome_assunto', 'codigo_orgao', 'nome_orgao']
Amostra: [{'ano': 2000, 'grau': 'G1', 'processo': '00482790520008050001', 'id_municipio': '2927408', 'municipio': '', 'codigo_classe': '100', 'nome_classe': 'Dúvida', 'codigo_assunto': '10301', 'nome_assunto': 'Férias', 'codigo_orgao': '17185', 'nome_orgao': '17ª VARA DE RELACOES DE CONSUMO'}, {'ano': 2000, 'grau': 'G1', 'processo': '00658265820008050001', 'id_municipio': '2927408', 'municipio': '', 'codigo_classe': '108', 'nome_classe': 'Falência de Empresários, Sociedades Empresáriais, Microempresas e Empresas de Pequeno Porte', 'codigo_assunto': '4998', 'nome_assunto': 'Autofalência', 'codigo_orgao': '17184', 'nome_orgao': '10ª VARA CIVEL E COMERCIAL'}, {'ano': 2000, 'grau': 'G1', 'processo': '0006612932